In [0]:
import sys
import os
import importlib

# 1. 'src' 폴더가 존재하는 최상위 루트 경로(maps)를 찾아 sys.path에 등록
# Serverless에서 os.getcwd()가 노트북 디렉터리가 아닐 수 있으므로 sys.path 우선 탐색
current_dir = next((p for p in sys.path if os.path.isdir(os.path.join(p, "src"))), None)
if current_dir is None:
    current_dir = os.getcwd()
    while current_dir != "/" and not os.path.exists(os.path.join(current_dir, "src")):
        current_dir = os.path.dirname(current_dir)

if os.path.exists(os.path.join(current_dir, "src")):
    if current_dir not in sys.path:
        sys.path.insert(0, current_dir)  # 최우선 탐색 경로로 등록
else:
    raise FileNotFoundError("루트 경로에서 'src' 디렉터리를 찾을 수 없습니다.")

# 2. 모듈 Import 및 강제 Reload
import src.dq.dq_config as dq_config
import src.dq.dq_functions as dq_functions
import src.dq.dq_runner as dq_runner

importlib.reload(dq_config)
importlib.reload(dq_functions)
importlib.reload(dq_runner)

from src.dq.dq_runner import DQRunner

print("DQ 모듈 로드 성공! (Root Path:", current_dir, ")")

In [0]:
# from pyspark.sql import functions as F

# # Spark 세션 시간대 설정
# spark.conf.set("spark.sql.session.timeZone", "Asia/Seoul")

# print(
#     "Spark Session Timezone:",
#     spark.conf.get("spark.sql.session.timeZone")
# )

In [0]:
import datetime

# 1. config 파일에서 DQ_RULES에 등록된 모든 대상 테이블 목록을 자동으로 추출 (채널 추가 시 자동 반영!)
from src.dq.dq_config import DQ_RULES, DQ_RESULT_TABLE
target_tables = list(set([rule["target_table"] for rule in DQ_RULES]))

print(f"📋 [자동 감지된 검증 대상 테이블 목록]: {target_tables}")

# 2. 러너 객체 생성
runner = DQRunner(spark=spark, dbutils=dbutils)

# 3. [중복 방지] 오늘 날짜 기준으로 기존 DQ 결과가 있다면 전체 채널 실행 전 딱 1번만 초기화
# (재실행 시 데이터가 중복으로 쌓이는 것을 방지합니다)
current_date = datetime.date.today().strftime("%Y-%m-%d")

try:
    spark.sql(f"DELETE FROM {DQ_RESULT_TABLE} WHERE ingest_date = '{current_date}'")
    print(f"🧹 [DQ 결과 초기화] {current_date} 일자 기준 기존 DQ 결과 정리를 완료했습니다.")
except Exception as e:
    print(f"ℹ️ [안내] 초기화 생략 (테이블이 아직 없거나 첫 실행): {e}")

# 4. 모든 채널/테이블을 자동으로 순회하며 DQ 검사 실행 및 결과 행(Row) 단위 누적(Append)
total_summaries = []

for target_table in target_tables:
    print(f"\n==================================================")
    print(f"🚀 [DQ 실행 중] 대상 테이블: {target_table}")
    print(f"==================================================s")
    
    # 각 테이블별 DQ 실행 (내부적으로 결과가 Append 됨)
    dq_results, dq_summary = runner.run_table_dq(target_table=target_table)
    
    total_summaries.append(dq_summary)
    print(f"✅ [완료] {target_table} 검증 완료")

print("\n🎉 [모든 채널 DQ 검사 완료] dq_result 테이블에 결과가 안전하게 누적되었습니다.")